# Connectivity 5 — Per-neuron P(AD) from the ISI MLP vs GLM connectivity

| | |
|---|---|
| **Original** | `Liam_work/tests/behavior_isi_analysis/P_AD_vs_Connectivity.ipynb` |
| **Status** | Side project (Apr 2026). Bridges the MLP and the connectivity work. |

Do neurons the ISI-histogram MLP finds "more AD-like" sit in less-connected parts of the REM GLM graph,
specifically in old AD animals? Re-runs a **leave-one-animal-out** version of the raw-ISI MLP (an older
setup: fixed architecture, no hyperopt, no oversampling — not identical to step 3) to get held-out
per-neuron P(AD), joins it positionally to GLM graph metrics (degree, strength, clustering, betweenness,
eigenvector centrality) within (animal, region, source epoch), then: per-animal Spearman, firing-rate
partial correlations, a mixed model `P(AD) ~ metric * geno * age_group + REM_FR + (1|animal)`, decile
tails, per-region breakdown and animal-level fallback. 5 animals are dropped because their GLM epoch
can't be matched.

Reads the 13 GB raw ISI table and `data/glm_results/glm_connectivity_results_20animals_REM.pkl`.

Paths: raw inputs are read from `/media/HlabShare/...` (read-only). Anything this copy writes goes under `/hlabhome/wg-mjames/git/LDCODES12_ind_neuron_spiking_classification_project/` — never back into `Liam_work/`.

---
*Repo copy: outputs stripped, header/section notes added, lab preamble added where missing, and output
paths moved into this repo. Analysis logic is unchanged from Liam's version.*

In [ ]:
# =============================================================================
# LAB-WIDE PREAMBLE — CPU thread limits + figure-export font config
# (added in the repo copy; set BEFORE numpy/scipy/sklearn/xgboost are imported)
# =============================================================================
import os
import matplotlib.pyplot as plt

plt.rcParams['pdf.fonttype'] = 42
plt.rcParams.update({'font.family': 'DejaVu Sans'})

# Limit CPU threads — shared server "marvin"
os.environ['OMP_NUM_THREADS']        = '4'
os.environ['OPENBLAS_NUM_THREADS']   = '4'
os.environ['MKL_NUM_THREADS']        = '4'
os.environ['VECLIB_MAXIMUM_THREADS'] = '4'
os.environ['NUMEXPR_NUM_THREADS']    = '4'

# P(AD) vs Functional Connectivity

**Question.** Do neurons the ISI-based MLP classifies as "more AD-like" sit in *less connected* neighborhoods of the REM functional connectivity graph? And is that relationship specific to **old AD** animals (symptomatic) vs old WT / young animals?

**Two independent views of each neuron:**
1. **P(AD)** — per-neuron probability from the Raw ISI MLP in `Raw_ISI_MLP_Classification.ipynb` (leave-one-animal-out held-out predictions; based only on ISI histogram shape).
2. **Functional connectivity** — directed, weighted coupling matrix from the L1 Point Process GLM in `AD_PointProcess_GLM_Connectivity.ipynb` (Pillow/Truccolo framework, REM sleep, within-region, raised-cosine basis).

**Merge key.** No stable neuron ID across files, so we match positionally within `(animal, brain_region, epoch)` after identifying the "source epoch" used by the GLM notebook (identified by matching region neuron counts).

**Graph metrics (NetworkX).** Degree, strength, clustering coefficient, betweenness, eigenvector centrality per neuron.

## 1. Imports & configuration

In [ ]:
import os
import time
import pickle
import warnings
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import accuracy_score, roc_auc_score
from imblearn.over_sampling import SMOTE
from joblib import Parallel, delayed

warnings.filterwarnings('ignore')
plt.style.use('seaborn-whitegrid')
np.random.seed(42)

# Paths
ISI_PKL = '/media/HlabShare/Ben_work/isi/all_animal_isi_df.pkl'
GLM_PKL = '/hlabhome/wg-mjames/git/LDCODES12_ind_neuron_spiking_classification_project/data/glm_results/glm_connectivity_results_20animals_REM.pkl'  # repo copy

# Config (mirrors Raw_ISI_MLP_Classification.ipynb)
AGE_CUTOFF       = 200
N_JOBS           = 20
EXCLUDE_ANIMALS  = ['caf104']   # 81% of neurons have within-epoch ISIs >100s
N_BINS           = 100
BIN_EDGES        = np.logspace(np.log10(0.001), np.log10(60), N_BINS + 1)

# The GLM pickle uses 'ACC'/'RSC' for some animals; ISI df uses 'ACad'/'RSPv'. Normalize.
REGION_MAP = {'ACC': 'ACad', 'RSC': 'RSPv'}
def norm_region(r):
    return REGION_MAP.get(r, r)

# Genotype colors (match existing notebooks)
GENO_COLORS = {'AD': '#C0392B', 'WT': '#2471A3'}

print(f'Config loaded. Using {N_JOBS} parallel workers.')

## 2. Load ISI dataframe

In [ ]:
print('Loading ISI dataframe (~13 GB — takes a moment)...')
t0 = time.time()
df = pd.read_pickle(ISI_PKL)

n_before = len(df)
df = df[~df['animal'].isin(EXCLUDE_ANIMALS)].reset_index(drop=True)
df['age_group']  = df['age'].apply(lambda x: 'OLD' if x > AGE_CUTOFF else 'YOUNG')
df['geno_label'] = df['geno'].map({'wt': 'WT', 'app_ps1': 'AD'})
df['group']      = df['geno_label'] + '_' + df['age_group']
df['label']      = (df['geno'] == 'app_ps1').astype(int)

print(f'Loaded in {time.time() - t0:.1f}s.')
print(f'Excluded {EXCLUDE_ANIMALS}: {n_before} -> {len(df)} neuron-rows')
print(f'Animals: {df["animal"].nunique()} | Regions: {sorted(df["brain_region"].unique())}')
df.head(3)

## 3. Load GLM connectivity pickle

In [ ]:
with open(GLM_PKL, 'rb') as f:
    glm = pickle.load(f)

print(f'Animals in GLM pickle: {len(glm)}')
print(f'Sample keys: {list(glm.keys())[:5]}')

example_key = list(glm.keys())[0]
example = glm[example_key]
print(f'\n--- Example ({example_key}) ---')
print(f'  keys              : {list(example.keys())}')
print(f'  genotype          : {example["genotype"]}')
print(f'  n_neurons         : {example["n_neurons"]}')
print(f'  regions           : {list(example["regions"].keys())}')
print(f'  neuron_info[0]    : {example["neuron_info"][0]}')
first_region = list(example['regions'].keys())[0]
print(f'  connectivity shape ({first_region}): {example["regions"][first_region]["connectivity"].shape}')

## 4. Compute per-neuron graph metrics with NetworkX

For each `(animal, region)`, treat the GLM coupling matrix as a directed weighted graph and extract per-neuron metrics.

**Metrics extracted:**
- `in_degree_u`, `out_degree_u`, `total_degree_u` — unweighted (count of nonzero couplings)
- `in_strength`, `out_strength`, `total_strength` — sum of coupling weights
- `clustering_coef` — undirected clustering (symmetrized graph)
- `betweenness` — directed betweenness centrality
- `eigenvector_centrality` — directed hub score

We pair each row with its position in the GLM's `neuron_indices` list so we can positionally match to the ISI dataframe later.

In [ ]:
def compute_graph_metrics_for_matrix(conn):
    """Return a dict of per-neuron graph metrics from a signed weighted directed matrix.

    `conn` is NxN, conn[i,j] is the coupling weight from i -> j.
    """
    n = conn.shape[0]
    # Use absolute weight for magnitude-based metrics; many centralities require positive weights.
    W = np.abs(conn.astype(float))
    # Binary mask for unweighted metrics
    B = (W > 0).astype(int)

    # Degree (unweighted)
    out_deg = B.sum(axis=1)          # outgoing edges from each node
    in_deg  = B.sum(axis=0)          # incoming edges to each node
    tot_deg = out_deg + in_deg

    # Strength (weighted)
    out_str = W.sum(axis=1)
    in_str  = W.sum(axis=0)
    tot_str = out_str + in_str

    # NetworkX graphs
    G_dir = nx.from_numpy_array(W, create_using=nx.DiGraph)
    G_und = nx.from_numpy_array((W + W.T) / 2.0, create_using=nx.Graph)

    # Clustering (undirected weighted)
    try:
        clust = nx.clustering(G_und, weight='weight')
        clust_arr = np.array([clust.get(i, 0.0) for i in range(n)])
    except Exception:
        clust_arr = np.zeros(n)

    # Betweenness (directed)
    try:
        bet = nx.betweenness_centrality(G_dir, weight='weight', normalized=True)
        bet_arr = np.array([bet.get(i, 0.0) for i in range(n)])
    except Exception:
        bet_arr = np.zeros(n)

    # Eigenvector centrality (directed; can fail on disconnected / no-edge graphs)
    try:
        eig = nx.eigenvector_centrality_numpy(G_dir, weight='weight', max_iter=200)
        eig_arr = np.array([eig.get(i, 0.0) for i in range(n)])
    except Exception:
        eig_arr = np.zeros(n)

    return {
        'in_degree_u': in_deg, 'out_degree_u': out_deg, 'total_degree_u': tot_deg,
        'in_strength': in_str, 'out_strength': out_str, 'total_strength': tot_str,
        'clustering_coef': clust_arr,
        'betweenness':     bet_arr,
        'eigenvector_centrality': eig_arr,
    }


rows = []
for animal_upper, ga in glm.items():
    ni = ga['neuron_info']
    for raw_region, rdata in ga['regions'].items():
        conn = rdata['connectivity']
        nidx = rdata['neuron_indices']
        if conn.shape[0] == 0:
            continue
        m = compute_graph_metrics_for_matrix(conn)
        for pos, info_idx in enumerate(nidx):
            info = ni[info_idx]
            rows.append({
                'animal_upper':   animal_upper,
                'brain_region':   norm_region(raw_region),
                'glm_region_raw': raw_region,
                'pos_in_region':  pos,
                'cell_type':      info['cell_type'],
                'glm_firing_rate_hz': info['firing_rate_hz'],
                'glm_n_spikes':   info['n_spikes'],
                **{k: v[pos] for k, v in m.items()}
            })

graph_metrics_df = pd.DataFrame(rows)
print(f'Graph metrics computed for {len(graph_metrics_df):,} neuron-region slots '
      f'across {graph_metrics_df["animal_upper"].nunique()} animals.')
graph_metrics_df.head()

## 5. Identify the GLM "source epoch" for each animal

The GLM pickle was built from ONE epoch per animal (`"first recording"`), but the ISI dataframe contains multiple epochs per animal. We identify the GLM source epoch for each animal by finding the ISI epoch whose per-region neuron counts **match** the GLM's per-region counts exactly.

From prior inspection: 15/20 animals have a clean matching epoch. The other 5 (BTH90/95/100, CAF42, ZBR113) will be flagged and dropped — their region counts never line up with the GLM's, so the positional-match assumption is unsafe.

In [ ]:
source_epoch = {}
for animal_upper, ga in glm.items():
    animal_lower = animal_upper.lower()
    isi_sub = df[df['animal'] == animal_lower]
    if len(isi_sub) == 0:
        continue

    # Region sizes in GLM (with region name normalization)
    glm_sizes = {norm_region(r): ga['regions'][r]['connectivity'].shape[0]
                 for r in ga['regions']}

    for ep in sorted(isi_sub['epoch'].unique()):
        isi_sizes = isi_sub[isi_sub['epoch'] == ep].groupby('brain_region').size().to_dict()
        if all(isi_sizes.get(r, -1) == c for r, c in glm_sizes.items()):
            source_epoch[animal_upper] = ep
            break

matched = sorted(source_epoch.keys())
unmatched = sorted(a for a in glm.keys() if a not in source_epoch)
print(f'MATCHED ({len(matched)}): {matched}')
print(f'UNMATCHED ({len(unmatched)}): {unmatched}')

## 6. Re-run the Raw ISI MLP to get per-neuron P(AD)

We reproduce the **All States / All Animals** LOAO pipeline from `Raw_ISI_MLP_Classification.ipynb` cells 3–5. Each neuron-row gets its out-of-fold `P(AD)` attached. (This is the held-out prediction — the animal the neuron belongs to was never in training.)

In [ ]:
# Histogram representation (identical to the Raw ISI notebook)

def clean_isi_array(isi_list, min_isi=0, max_isi=60):
    all_isis = []
    for arr in isi_list:
        if isinstance(arr, np.ndarray) and len(arr) > 0:
            valid = arr[(arr > min_isi) & (arr <= max_isi)]
            all_isis.extend(valid)
    return np.array(all_isis)


def isi_to_histogram(isi_list, bin_edges=BIN_EDGES):
    isis = clean_isi_array(isi_list)
    if len(isis) < 10:
        return np.zeros(len(bin_edges) - 1)
    hist, _ = np.histogram(isis, bins=bin_edges)
    hist = hist.astype(float)
    hmin, hmax = hist.min(), hist.max()
    if hmax > hmin:
        hist = (hist - hmin) / (hmax - hmin)
    elif hmax > 0:
        hist = np.ones_like(hist)
    return hist


def _process_row(row):
    rec = {
        'animal':     row['animal'],
        'epoch':      row['epoch'],
        'cell_no':    row['cell_no'],
        'brain_region': row['brain_region'],
        'cell_type':  row['cell_type'],
        'age':        row['age'],
        'age_group':  row['age_group'],
        'geno_label': row['geno_label'],
        'peak_latency': row['peak_latency'],
        'label':      row['label'],
    }
    for state in ['wake', 'nrem', 'rem']:
        hist = isi_to_histogram(row[f'{state}_isi'])
        for b in range(len(hist)):
            rec[f'{state}_bin{b}'] = hist[b]
        # Also compute firing rate per state (spikes / cumulative ISI time)
        isis = clean_isi_array(row[f'{state}_isi'])
        if len(isis) > 0 and isis.sum() > 0:
            rec[f'{state}_fr_hz'] = len(isis) / isis.sum()
        else:
            rec[f'{state}_fr_hz'] = np.nan
    return rec


print('Building histogram + firing-rate features...')
t0 = time.time()
records = Parallel(n_jobs=N_JOBS, verbose=1)(
    delayed(_process_row)(r) for _, r in df.iterrows()
)
hist_df = pd.DataFrame(records)
feat_cols = [c for c in hist_df.columns if '_bin' in c]
print(f'Histogram features built in {time.time() - t0:.1f}s. '
      f'n_features={len(feat_cols)}, n_rows={len(hist_df):,}')

In [ ]:
# LOAO fold worker — mirrors Raw_ISI_MLP_Classification.ipynb

def _train_one_fold(train_idx, test_idx, X, y, groups, hidden_layers):
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X[train_idx])
    X_test  = scaler.transform(X[test_idx])
    y_train = y[train_idx]

    minority_count = min(int(np.sum(y_train == 1)), int(np.sum(y_train == 0)))
    k = min(5, minority_count - 1)
    if k >= 1 and minority_count >= 2:
        X_train, y_train = SMOTE(k_neighbors=k, random_state=42).fit_resample(X_train, y_train)

    mlp = MLPClassifier(
        hidden_layer_sizes=hidden_layers, activation='relu', solver='adam',
        alpha=0.01, batch_size=256, learning_rate='adaptive', learning_rate_init=0.001,
        max_iter=300, early_stopping=True, validation_fraction=0.15,
        n_iter_no_change=10, random_state=42,
    )
    mlp.fit(X_train, y_train)
    probs = mlp.predict_proba(X_test)[:, 1]
    preds = (probs > 0.5).astype(int)
    return test_idx, preds, probs


X = hist_df[feat_cols].values.astype(float)
y = hist_df['label'].values
groups = hist_df['animal'].values

# Fill NaNs with column medians (only affects empty-state neurons)
for j in range(X.shape[1]):
    m = np.isnan(X[:, j])
    if m.any():
        med = np.nanmedian(X[:, j])
        X[m, j] = 0.0 if np.isnan(med) else med

logo = LeaveOneGroupOut()
folds = list(logo.split(X, y, groups))
print(f'Running {len(folds)} LOAO folds in parallel...')
t0 = time.time()
fold_out = Parallel(n_jobs=N_JOBS)(
    delayed(_train_one_fold)(tr, te, X, y, groups, (256, 128, 64)) for tr, te in folds
)
print(f'LOAO done in {time.time() - t0:.1f}s.')

P_AD = np.zeros(len(y))
Y_PRED = np.zeros(len(y), dtype=int)
for test_idx, preds, probs in fold_out:
    P_AD[test_idx]  = probs
    Y_PRED[test_idx] = preds

hist_df['P_AD']   = P_AD
hist_df['y_pred'] = Y_PRED

# Sanity checks
neuron_acc = accuracy_score(y, Y_PRED)
neuron_auc = roc_auc_score(y, P_AD)
animal_acc = np.mean([
    (hist_df.loc[hist_df['animal'] == a, 'P_AD'].mean() > 0.5)
    == bool(hist_df.loc[hist_df['animal'] == a, 'label'].iloc[0])
    for a in hist_df['animal'].unique()
])
print(f'Neuron Acc = {neuron_acc:.1%} | Neuron AUC = {neuron_auc:.3f} | Animal Acc = {animal_acc:.1%}')

## 7. Build per-neuron lookup (ISI side)

Select the single row per neuron corresponding to the GLM source epoch for each animal, sorted by `cell_no`. This gives 1-to-1 positional correspondence to `graph_metrics_df` within `(animal, region)`.

In [ ]:
# Keep only rows whose (animal_upper, epoch) matches the GLM source epoch
hist_df['animal_upper'] = hist_df['animal'].str.upper()
isi_match_rows = []
for animal_upper, ep in source_epoch.items():
    sub = hist_df[(hist_df['animal_upper'] == animal_upper) & (hist_df['epoch'] == ep)].copy()
    sub = sub.sort_values(['brain_region', 'cell_no']).reset_index(drop=True)
    sub['pos_in_region'] = sub.groupby('brain_region').cumcount()
    isi_match_rows.append(sub)

isi_matched = pd.concat(isi_match_rows, ignore_index=True)
print(f'ISI rows retained for matched animals/epochs: {len(isi_matched):,}')
isi_matched.groupby(['animal_upper', 'brain_region']).size().head(12)

## 8. Join ISI ⇄ graph metrics and verify

Merge on `(animal_upper, brain_region, pos_in_region)`. Then **verify** the merge by checking that GLM `cell_type` and firing rate match the ISI row's — proof that the positional assumption held. Any mismatches get flagged.

In [ ]:
merged = isi_matched.merge(
    graph_metrics_df,
    on=['animal_upper', 'brain_region', 'pos_in_region'],
    how='inner',
    suffixes=('_isi', '_glm'),
)
print(f'Merged rows: {len(merged):,}')

# Verification: cell_type must agree; firing rates should be very close
ct_mismatch = (merged['cell_type_isi'] != merged['cell_type_glm']).sum()
fr_diff = (merged['rem_fr_hz'] - merged['glm_firing_rate_hz']).abs()
fr_diff_ok = (fr_diff < 0.5).sum()  # < 0.5 Hz difference is fine
print(f'  cell_type mismatches: {ct_mismatch}')
print(f'  firing-rate |diff| < 0.5 Hz: {fr_diff_ok}/{len(merged)}')
print(f'  median |fr_diff|: {fr_diff.median():.3f} Hz, max: {fr_diff.max():.3f} Hz')

# Per-animal match report
print('\n--- Per-animal merge report ---')
report = (merged.groupby(['animal_upper', 'geno_label', 'age_group'])
                .size().reset_index(name='n_merged'))
print(report.to_string(index=False))

## 9. Firing-rate confound check

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
metrics = ['total_degree_u', 'total_strength', 'clustering_coef',
           'betweenness', 'eigenvector_centrality', 'in_strength']
for ax, mcol in zip(axes.flat, metrics):
    for g, color in GENO_COLORS.items():
        sub = merged[merged['geno_label'] == g]
        ax.scatter(sub['rem_fr_hz'], sub[mcol],
                   s=6, alpha=0.35, color=color, label=g)
    ax.set_xlabel('REM firing rate (Hz)')
    ax.set_ylabel(mcol)
    ax.set_xscale('log')
    ax.legend(fontsize=8)
    r, _ = stats.spearmanr(merged['rem_fr_hz'].fillna(0), merged[mcol].fillna(0))
    ax.set_title(f'{mcol}  (Spearman ρ = {r:.2f})', fontsize=10)
fig.suptitle('Graph metric vs REM firing rate — confound check', fontweight='bold')
plt.tight_layout()
plt.show()

## 10. Primary analysis — per-animal Spearman correlations between P(AD) and each graph metric

For each animal we compute ρ between `P(AD)` and each graph metric across its neurons. That gives one ρ per animal per metric. Then we compare the distribution of ρ between AD and WT (Mann–Whitney), split by age group.

In [ ]:
GRAPH_METRICS = ['total_degree_u', 'total_strength',
                 'in_degree_u', 'out_degree_u', 'in_strength', 'out_strength',
                 'clustering_coef', 'betweenness', 'eigenvector_centrality']

CORR_COLS = ['animal', 'geno', 'age_group', 'age', 'n', 'rho', 'p']


def per_animal_corr(merged, metric, method='spearman'):
    rows = []
    for a, sub in merged.groupby('animal_upper'):
        if len(sub) < 10 or sub[metric].nunique() < 3:
            continue
        fn = stats.spearmanr if method == 'spearman' else stats.pearsonr
        rho, p = fn(sub['P_AD'], sub[metric])
        rows.append({
            'animal':     a,
            'geno':       sub['geno_label'].iloc[0],
            'age_group':  sub['age_group'].iloc[0],
            'age':        sub['age'].iloc[0],
            'n':          len(sub),
            'rho':        rho,
            'p':          p,
        })
    return pd.DataFrame(rows, columns=CORR_COLS)


def plot_rho_strip(ax, c, title):
    # Plot per-animal rho strip; safe when c is empty or has missing groups.
    if len(c) == 0:
        ax.text(0.5, 0.5, '(no animals met variance threshold)',
                ha='center', va='center', transform=ax.transAxes,
                fontsize=9, color='gray')
        ax.set_title(title, fontsize=10)
        return
    sns.stripplot(data=c, x='age_group', y='rho', hue='geno',
                  order=['YOUNG', 'OLD'], hue_order=['WT', 'AD'],
                  palette=GENO_COLORS, dodge=True, size=9, ax=ax,
                  edgecolor='black', linewidth=0.5)
    ax.axhline(0, color='black', lw=0.8, alpha=0.6)
    ax.set_title(title, fontsize=10)
    ax.set_xlabel('')
    if ax.get_legend() is not None:
        ax.get_legend().remove()


fig, axes = plt.subplots(3, 3, figsize=(14, 11), sharey=True)
summary_rows = []

for ax, metric in zip(axes.flat, GRAPH_METRICS):
    c = per_animal_corr(merged, metric)
    plot_rho_strip(ax, c, metric)
    ax.set_ylabel('Spearman ρ (P(AD) vs metric, per animal)')

    for ag in ['YOUNG', 'OLD']:
        wt = c[(c['age_group'] == ag) & (c['geno'] == 'WT')]['rho'].values
        ad = c[(c['age_group'] == ag) & (c['geno'] == 'AD')]['rho'].values
        if len(wt) >= 2 and len(ad) >= 2:
            u, p = stats.mannwhitneyu(wt, ad, alternative='two-sided')
            summary_rows.append({
                'metric':   metric,
                'age_group': ag,
                'n_WT':     len(wt), 'n_AD': len(ad),
                'median_WT_rho': np.median(wt),
                'median_AD_rho': np.median(ad),
                'U':        u,
                'p':        p,
            })

fig.suptitle('Per-animal Spearman ρ: P(AD) vs graph metric', fontweight='bold')
handles = [plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=GENO_COLORS[g],
                       markersize=10, label=g, markeredgecolor='black')
           for g in ['WT', 'AD']]
fig.legend(handles=handles, loc='lower center', ncol=2, bbox_to_anchor=(0.5, -0.02))
plt.tight_layout()
plt.show()

summary_df = pd.DataFrame(summary_rows)
print('\n=== Mann-Whitney (WT vs AD per age-group) ===')
if len(summary_df):
    print(summary_df.sort_values(['age_group', 'p']).to_string(index=False))
else:
    print('(no metrics had enough animals per group for the test)')


## 11. Partial correlation — controlling for REM firing rate

Lower-firing neurons are both noisier ISI-wise (→ more uncertain P(AD)) and less likely to have detectable connections (→ lower degree). We residualize **both** P(AD) and the graph metric against REM firing rate **within each animal**, then correlate the residuals.

In [ ]:
PARTIAL_COLS = ['animal', 'geno', 'age_group', 'n', 'rho', 'p']


def residualize(y, x):
    # Return y minus linear regression on x. NaN-safe.
    y = np.asarray(y, dtype=float); x = np.asarray(x, dtype=float)
    mask = np.isfinite(y) & np.isfinite(x)
    if mask.sum() < 3:
        return np.full_like(y, np.nan)
    slope, intercept = np.polyfit(x[mask], y[mask], 1)
    out = y - (slope * x + intercept)
    out[~mask] = np.nan
    return out


def per_animal_partial(merged, metric):
    rows = []
    for a, sub in merged.groupby('animal_upper'):
        if len(sub) < 10 or sub[metric].nunique() < 3:
            continue
        fr = sub['rem_fr_hz'].values
        pad_res = residualize(sub['P_AD'].values, fr)
        met_res = residualize(sub[metric].values, fr)
        mask = np.isfinite(pad_res) & np.isfinite(met_res)
        if mask.sum() < 10:
            continue
        rho, p = stats.spearmanr(pad_res[mask], met_res[mask])
        rows.append({
            'animal':    a,
            'geno':      sub['geno_label'].iloc[0],
            'age_group': sub['age_group'].iloc[0],
            'n':         int(mask.sum()),
            'rho':       rho,
            'p':         p,
        })
    return pd.DataFrame(rows, columns=PARTIAL_COLS)


fig, axes = plt.subplots(3, 3, figsize=(14, 11), sharey=True)
partial_summary = []

for ax, metric in zip(axes.flat, GRAPH_METRICS):
    c = per_animal_partial(merged, metric)
    plot_rho_strip(ax, c, metric)
    ax.set_ylabel('Partial Spearman ρ (firing-rate controlled)')

    for ag in ['YOUNG', 'OLD']:
        wt = c[(c['age_group'] == ag) & (c['geno'] == 'WT')]['rho'].values
        ad = c[(c['age_group'] == ag) & (c['geno'] == 'AD')]['rho'].values
        if len(wt) >= 2 and len(ad) >= 2:
            u, p = stats.mannwhitneyu(wt, ad, alternative='two-sided')
            partial_summary.append({
                'metric':   metric,
                'age_group': ag,
                'n_WT':     len(wt), 'n_AD': len(ad),
                'median_WT_rho': np.median(wt),
                'median_AD_rho': np.median(ad),
                'U':        u,
                'p':        p,
            })

fig.suptitle('Per-animal Spearman ρ (PARTIAL, firing-rate controlled)', fontweight='bold')
handles = [plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=GENO_COLORS[g],
                       markersize=10, label=g, markeredgecolor='black')
           for g in ['WT', 'AD']]
fig.legend(handles=handles, loc='lower center', ncol=2, bbox_to_anchor=(0.5, -0.02))
plt.tight_layout()
plt.show()

partial_summary_df = pd.DataFrame(partial_summary)
print('\n=== Partial-correlation Mann-Whitney ===')
if len(partial_summary_df):
    print(partial_summary_df.sort_values(['age_group', 'p']).to_string(index=False))
else:
    print('(no metrics had enough animals per group for the test)')


## 12. Mixed-effects model (primary statistical result)

For each metric we fit:
```
P(AD) ~ metric * geno * age_group + REM_FR + (1 | animal)
```
The `metric:geno:age_group` three-way interaction is the formal test of *"does connectivity predict P(AD) differently in old-AD vs everyone else?"*.

Within-animal rank-normalization stabilizes scales before fitting.

In [ ]:
import statsmodels.formula.api as smf

def rank_within_animal(df, cols):
    out = df.copy()
    for c in cols:
        out[c + '_rank'] = out.groupby('animal_upper')[c].rank(pct=True)
    return out


merged_r = rank_within_animal(
    merged,
    ['P_AD', 'rem_fr_hz'] + GRAPH_METRICS,
)

mixed_rows = []
for metric in GRAPH_METRICS:
    m_col = metric + '_rank'
    # Model: rank-normalized P_AD vs rank-normalized metric with genotype and age-group interactions
    formula = (f'P_AD_rank ~ {m_col} * C(geno_label) * C(age_group) '
               f'+ rem_fr_hz_rank')
    try:
        mdl = smf.mixedlm(formula, data=merged_r, groups=merged_r['animal_upper']).fit(
            reml=False, method='lbfgs'
        )
        pvals = mdl.pvalues
        # Find the three-way interaction term(s) — pattern depends on encoding
        threeway = [k for k in pvals.index
                    if m_col in k and 'geno_label' in k and 'age_group' in k]
        coef_threeway = {k: (mdl.params[k], pvals[k]) for k in threeway}
        mixed_rows.append({
            'metric':      metric,
            'converged':   mdl.converged,
            'n_obs':       int(mdl.nobs),
            'main_metric_coef': mdl.params.get(m_col, np.nan),
            'main_metric_p':    pvals.get(m_col, np.nan),
            'threeway':    coef_threeway,
        })
    except Exception as e:
        mixed_rows.append({'metric': metric, 'converged': False, 'error': str(e)})

mix_df = pd.DataFrame(mixed_rows)
print('=== Mixed-effects model summary ===')
for _, r in mix_df.iterrows():
    print(f'\n{r["metric"]}  (converged={r.get("converged", False)}, n={r.get("n_obs","?")})')
    if 'error' in r and isinstance(r.get('error'), str):
        print(f'  ERROR: {r["error"]}')
        continue
    print(f'  main metric coef = {r["main_metric_coef"]:.4f}, p = {r["main_metric_p"]:.4g}')
    for k, (coef, p) in (r['threeway'] or {}).items():
        print(f'  {k}: coef={coef:.4f}, p={p:.4g}')

## 13. Tail analysis — top vs bottom decile of P(AD)

Correlations can hide non-linear effects. Within each animal we compare graph metrics between the **top 10%** and **bottom 10%** of neurons ranked by P(AD).

If the hypothesis holds for old-AD animals, the top-decile (highest P(AD)) neurons should have **lower** strength/degree than the bottom-decile.

In [ ]:
tail_rows = []
for a, sub in merged.groupby('animal_upper'):
    if len(sub) < 20:
        continue
    q10, q90 = sub['P_AD'].quantile([0.1, 0.9])
    low = sub[sub['P_AD'] <= q10]
    high = sub[sub['P_AD'] >= q90]
    for metric in GRAPH_METRICS:
        tail_rows.append({
            'animal':    a,
            'geno':      sub['geno_label'].iloc[0],
            'age_group': sub['age_group'].iloc[0],
            'metric':    metric,
            'low_tail_median':  low[metric].median(),
            'high_tail_median': high[metric].median(),
            'diff_high_minus_low': high[metric].median() - low[metric].median(),
        })
tail_df = pd.DataFrame(tail_rows)

fig, axes = plt.subplots(3, 3, figsize=(14, 11), sharey=False)
for ax, metric in zip(axes.flat, GRAPH_METRICS):
    sub = tail_df[tail_df['metric'] == metric]
    sns.stripplot(data=sub, x='age_group', y='diff_high_minus_low', hue='geno',
                  order=['YOUNG', 'OLD'], hue_order=['WT', 'AD'],
                  palette=GENO_COLORS, dodge=True, size=9, ax=ax,
                  edgecolor='black', linewidth=0.5)
    ax.axhline(0, color='black', lw=0.8, alpha=0.6)
    ax.set_title(metric, fontsize=10)
    ax.set_ylabel('median(top10% P_AD) - median(bot10%)')
    ax.set_xlabel('')
    ax.legend([], frameon=False)

fig.suptitle('Tail analysis: graph-metric difference between high-P(AD) and low-P(AD) neurons',
             fontweight='bold')
handles = [plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=GENO_COLORS[g],
                       markersize=10, label=g, markeredgecolor='black')
           for g in ['WT', 'AD']]
fig.legend(handles=handles, loc='lower center', ncol=2, bbox_to_anchor=(0.5, -0.02))
plt.tight_layout()
plt.show()

## 14. Per-region breakdown

In [ ]:
region_rows = []
for region in ['ACad', 'CA1', 'RSPv', 'V1']:
    reg_merged = merged[merged['brain_region'] == region]
    for metric in ['total_degree_u', 'total_strength', 'clustering_coef']:
        c = per_animal_partial(reg_merged, metric)
        for ag in ['YOUNG', 'OLD']:
            wt = c[(c['age_group'] == ag) & (c['geno'] == 'WT')]['rho'].values
            ad = c[(c['age_group'] == ag) & (c['geno'] == 'AD')]['rho'].values
            if len(wt) >= 2 and len(ad) >= 2:
                u, p = stats.mannwhitneyu(wt, ad, alternative='two-sided')
                region_rows.append({
                    'region':    region, 'metric': metric, 'age_group': ag,
                    'n_WT':      len(wt), 'n_AD': len(ad),
                    'median_WT': np.median(wt), 'median_AD': np.median(ad),
                    'p':         p,
                })
region_df = pd.DataFrame(region_rows)
print('=== Per-region partial-correlation contrasts ===')
print(region_df.sort_values(['region', 'age_group', 'p']).to_string(index=False))

## 15. Example visualization — one old-AD vs one old-WT animal

Pick the old-AD animal and old-WT animal with the strongest per-animal correlations. Show their coupling matrices reordered by P(AD) and the network graph colored by P(AD) / sized by strength.

In [ ]:
old_merged = merged[merged['age_group'] == 'OLD']
per_animal = per_animal_partial(old_merged, 'total_strength')
try:
    target_ad = per_animal[per_animal['geno']=='AD'].sort_values('rho').iloc[0]['animal']
    target_wt = per_animal[per_animal['geno']=='WT'].sort_values('rho', ascending=False).iloc[0]['animal']
except IndexError:
    target_ad = target_wt = None

print(f'AD example: {target_ad}  |  WT example: {target_wt}')

def draw_animal(ax_mat, ax_net, animal_upper, title_suffix):
    if animal_upper is None:
        ax_mat.set_title('(no data)'); return
    ga = glm[animal_upper]
    # Pick the largest region to visualize
    sizes = {r: ga['regions'][r]['connectivity'].shape[0] for r in ga['regions']}
    region = max(sizes, key=sizes.get)
    conn = np.abs(ga['regions'][region]['connectivity'])
    sub = merged[(merged['animal_upper'] == animal_upper)
                 & (merged['brain_region'] == norm_region(region))]
    sub = sub.sort_values('pos_in_region').reset_index(drop=True)
    order = np.argsort(sub['P_AD'].values)
    conn_ord = conn[np.ix_(order, order)]

    im = ax_mat.imshow(conn_ord, cmap='magma', aspect='auto')
    ax_mat.set_title(f'{animal_upper} {region}\n{title_suffix}\nreordered by P(AD) →', fontsize=10)
    plt.colorbar(im, ax=ax_mat, fraction=0.045)

    # Network graph
    G = nx.from_numpy_array(conn, create_using=nx.DiGraph)
    pos = nx.spring_layout(G, seed=7, weight='weight')
    strength = conn.sum(axis=0) + conn.sum(axis=1)
    node_sizes = 40 + 400 * (strength / strength.max() if strength.max() > 0 else np.ones_like(strength))
    node_colors = sub['P_AD'].values
    nx.draw_networkx_edges(G, pos, ax=ax_net, alpha=0.08, arrows=False, width=0.5)
    sc = nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=node_sizes,
                                cmap='coolwarm', vmin=0, vmax=1, ax=ax_net, edgecolors='black', linewidths=0.3)
    ax_net.set_title(f'Node size ∝ strength, color = P(AD)', fontsize=10)
    ax_net.axis('off')
    plt.colorbar(sc, ax=ax_net, fraction=0.045, label='P(AD)')


fig, axes = plt.subplots(2, 2, figsize=(14, 12))
draw_animal(axes[0, 0], axes[0, 1], target_ad, 'old AD')
draw_animal(axes[1, 0], axes[1, 1], target_wt, 'old WT')
plt.tight_layout()
plt.show()

## 16. Summary & interpretation

**What a positive finding looks like:**
- In Cell 10/11: old-AD animals have systematically *negative* ρ (less connected → higher P(AD)) that's closer to zero / positive in old-WT.
- In Cell 12: the `metric:geno:age_group` three-way interaction is significant for one or more metrics after firing-rate control.
- In Cell 13: top-decile P(AD) neurons have *lower* strength than bottom-decile, specifically in old-AD animals.

**What to be cautious about:**
- REM-only connectivity — the claim is only about REM coupling, not global anatomical connectivity.
- Within-region only — cannot speak to long-range / cross-region degeneration.
- 15/20 animals used for the per-neuron join; re-inspect with the 5 dropped animals (BTH90/95/100, CAF42, ZBR113) once their data lineage is reconciled.
- Multiple testing — if any p-value seems compelling, apply Benjamini–Hochberg FDR across the full metric × age × region panel.

**Robustness fallback (Cell 17):** animal-level analysis — 20 dots, one per animal — corroborates the per-neuron story without the join ambiguity.

## 17. Aggregate fallback — animal-level view

In [ ]:
animal_agg = (merged.groupby(['animal_upper', 'geno_label', 'age_group'])
                     .agg(mean_P_AD=('P_AD', 'mean'),
                          **{m: (m, 'mean') for m in GRAPH_METRICS})
                     .reset_index())

fig, axes = plt.subplots(3, 3, figsize=(14, 11))
for ax, m in zip(axes.flat, GRAPH_METRICS):
    for ag, marker in [('YOUNG', 'o'), ('OLD', '^')]:
        for g, color in GENO_COLORS.items():
            sub = animal_agg[(animal_agg['age_group'] == ag) & (animal_agg['geno_label'] == g)]
            ax.scatter(sub[m], sub['mean_P_AD'], s=130, marker=marker,
                       color=color, edgecolor='black', alpha=0.9,
                       label=f'{g} {ag}')
    ax.set_xlabel(m); ax.set_ylabel('animal mean P(AD)')
    ax.axhline(0.5, color='black', lw=0.6, alpha=0.4)
    ax.set_title(m, fontsize=10)

handles = [plt.Line2D([0], [0], marker=mk, color='w', markerfacecolor=GENO_COLORS[g],
                       markersize=11, label=f'{g} {ag}', markeredgecolor='black')
           for g in ['WT', 'AD'] for ag, mk in [('YOUNG', 'o'), ('OLD', '^')]]
fig.legend(handles=handles, loc='lower center', ncol=4, bbox_to_anchor=(0.5, -0.02))
fig.suptitle('Animal-level fallback: mean graph metric vs mean P(AD)', fontweight='bold')
plt.tight_layout()
plt.show()
print('\nAnimal-level dataframe:')
print(animal_agg.to_string(index=False))

## Per-mouse scatter — largest region only

One panel per animal. X = connectivity (in_strength), Y = P(AD). Restricted to the region with the most neurons so we're comparing apples to apples.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import spearmanr

# pick the region with the most neurons in the merged df
region_counts = merged['brain_region'].value_counts()
print("Neurons per region:")
print(region_counts)
top_region = region_counts.index[0]
print(f"\nUsing region: {top_region}")

sub_all = merged[merged['brain_region'] == top_region]
animals = sorted(sub_all['animal_upper'].unique())
n = len(animals)
ncols = 4
nrows = int(np.ceil(n / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(4*ncols, 3.2*nrows))
axes = axes.flatten()

for i, animal in enumerate(animals):
    ax = axes[i]
    sub = sub_all[sub_all['animal_upper'] == animal]
    geno = sub['geno_label'].iloc[0]
    age = sub['age_group'].iloc[0]
    x = sub['in_strength'].values
    y = sub['P_AD'].values
    ax.scatter(x, y, s=14, alpha=0.6, color=GENO_COLORS[geno], edgecolors='none')
    if len(sub) >= 5 and np.std(x) > 0:
        b, a = np.polyfit(x, y, 1)
        xs = np.linspace(x.min(), x.max(), 50)
        ax.plot(xs, a + b*xs, color='black', lw=1.5)
        rho, pval = spearmanr(x, y)
        ax.text(0.02, 0.98, f'ρ={rho:.2f}\np={pval:.2f}',
                transform=ax.transAxes, va='top', ha='left', fontsize=9,
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.7, edgecolor='none'))
    ax.set_title(f'{animal} ({geno}-{age}, n={len(sub)})', fontsize=10)
    ax.set_xlabel('in_strength')
    ax.set_ylabel('P(AD)')
    ax.set_ylim(-0.05, 1.05)

for j in range(n, len(axes)):
    axes[j].axis('off')

plt.suptitle(f'P(AD) vs in_strength — per mouse, {top_region} only', y=1.00, fontsize=13)
plt.tight_layout()
plt.show()


## Average connection strength per neuron vs P(AD)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import spearmanr

# mean strength per neuron = total_strength / total_degree (avg weight of its existing connections)
m = merged.copy()
m['mean_conn_strength'] = np.where(m['total_degree_u'] > 0,
                                   m['total_strength'] / m['total_degree_u'],
                                   np.nan)
m = m.dropna(subset=['mean_conn_strength'])

fig, ax = plt.subplots(figsize=(8, 6))
for geno in ['AD', 'WT']:
    sub = m[m['geno_label'] == geno]
    ax.scatter(sub['mean_conn_strength'], sub['P_AD'],
               s=14, alpha=0.5, color=GENO_COLORS[geno],
               label=f'{geno} (n={len(sub)})', edgecolors='none')

rho, pval = spearmanr(m['mean_conn_strength'], m['P_AD'])
ax.set_xlabel('Average strength of a neuron\'s connections')
ax.set_ylabel('P(AD)')
ax.set_title(f'Per-neuron: mean connection strength vs P(AD)\nSpearman ρ={rho:.3f}, p={pval:.2e}, n={len(m)}')
ax.legend()
plt.tight_layout()
plt.show()


## Direct-from-matrix version (no NetworkX)

For each neuron, go to the raw GLM coupling matrix, find every nonzero coupling it participates in (both incoming and outgoing), take the absolute value, and average. That's "average strength of this neuron's connections" — pure arithmetic on the matrix.

Then plot vs P(AD). One dot per neuron.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

# Re-walk the raw GLM pickle; for each neuron compute mean |weight| over its nonzero couplings
rows = []
for animal_upper, adata in glm.items():
    for region_name_raw, rdata in adata['regions'].items():
        region = norm_region(region_name_raw)
        conn = rdata['connectivity']            # N x N
        neuron_idx_list = rdata['neuron_indices']  # positions into adata['neuron_info']
        N = conn.shape[0]
        for local_i in range(N):
            row = conn[local_i, :].copy()
            col = conn[:, local_i].copy()
            # ignore self-loop
            row[local_i] = 0.0
            col[local_i] = 0.0
            # collect all nonzero couplings (both directions)
            weights = np.concatenate([row[row != 0], col[col != 0]])
            if len(weights) == 0:
                mean_abs_strength = np.nan
                n_connections = 0
            else:
                mean_abs_strength = float(np.mean(np.abs(weights)))
                n_connections = int(len(weights))
            rows.append({
                'animal_upper': animal_upper,
                'brain_region': region,
                'glm_pos': neuron_idx_list[local_i],
                'n_connections_raw': n_connections,
                'mean_abs_strength_raw': mean_abs_strength,
            })

raw_df = pd.DataFrame(rows)
print(f"Raw per-neuron rows: {len(raw_df)}")
print(raw_df.head())

# Attach to merged via (animal_upper, brain_region, pos_in_region)
# pos_in_region in `merged` is the order within each region after sort_values.
# raw_df.glm_pos is the position within neuron_info (animal-level). We need to align the two.
# In Cell 8 / Cell 15 the merge used sort_values(['brain_region','cell_no']) + cumcount() to get pos_in_region,
# and matched to neurons in the order they appear in each region of the GLM. That same ordering is what we used
# to build graph_metrics_df. Replicate it here: within each (animal, region) of raw_df, sort by glm_pos ascending
# and number 0..k-1.
raw_df = raw_df.sort_values(['animal_upper', 'brain_region', 'glm_pos']).reset_index(drop=True)
raw_df['pos_in_region'] = raw_df.groupby(['animal_upper', 'brain_region']).cumcount()

merged_raw = merged.merge(
    raw_df[['animal_upper', 'brain_region', 'pos_in_region',
            'n_connections_raw', 'mean_abs_strength_raw']],
    on=['animal_upper', 'brain_region', 'pos_in_region'],
    how='left'
)

# drop neurons with zero connections (no data point for "mean strength")
plot_df = merged_raw.dropna(subset=['mean_abs_strength_raw']).copy()
plot_df = plot_df[plot_df['n_connections_raw'] > 0]

print(f"\nNeurons with at least 1 connection: {len(plot_df)} / {len(merged_raw)}")

fig, ax = plt.subplots(figsize=(8, 6))
for geno in ['AD', 'WT']:
    sub = plot_df[plot_df['geno_label'] == geno]
    ax.scatter(sub['mean_abs_strength_raw'], sub['P_AD'],
               s=14, alpha=0.5, color=GENO_COLORS[geno],
               label=f'{geno} (n={len(sub)})', edgecolors='none')

rho, pval = spearmanr(plot_df['mean_abs_strength_raw'], plot_df['P_AD'])
ax.set_xlabel('Mean |coupling weight| over this neuron\'s nonzero connections (raw from matrix)')
ax.set_ylabel('P(AD)')
ax.set_title(f'Per-neuron: mean connection strength (direct from GLM matrix) vs P(AD)\n'
             f'Spearman ρ={rho:.3f}, p={pval:.2e}, n={len(plot_df)}')
ax.legend()
plt.tight_layout()
plt.show()
